# 05 - Ingestão Silver: Aplicação das Regras de Qualidade
**Squad 1 — Data Quality em Tempo Real | Dupla 1**
**Integrantes:** Gabriel Franz Simoni & Carlos Eduardo Santos de Souza
**Tabelas:** `ecommerce_itens_pedido` e `ecommerce_rastreamento_entregas`

### Objetivo (Card: Ingestão Silver):
Ler o micro-lote mais recente da camada Bronze, aplicar as 10 regras de qualidade definidas (5 técnicas + 5 de negócio) por tabela, adicionando uma coluna booleana de pass/fail por regra, e gravar o resultado na camada Silver com o metadado `silver_processed_at`.

**Pré-requisito:** `04_bronze_ingestao_delta.ipynb` já executado (Bronze de `ecommerce_itens_pedido` e `ecommerce_rastreamento_entregas` populada).

In [0]:
adls_client_id = dbutils.secrets.get(scope="internship-squad1", key="adls-client-id")
adls_tenant_id = dbutils.secrets.get(scope="internship-squad1", key="adls-tenant-id")
adls_client_secret = dbutils.secrets.get(scope="internship-squad1", key="adls-client-secret")
adls_storage_account = dbutils.secrets.get(scope="internship-squad1", key="adls-storage-account")

adls_options = {
    f"fs.azure.account.auth.type.{adls_storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{adls_storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{adls_storage_account}.dfs.core.windows.net": adls_client_id,
    f"fs.azure.account.oauth2.client.secret.{adls_storage_account}.dfs.core.windows.net": adls_client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{adls_storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{adls_tenant_id}/oauth2/token",
}

print("Credenciais OAuth preparadas.")

In [0]:
base_bronze = f"abfss://squad1@{adls_storage_account}.dfs.core.windows.net/dupla1/bronze"
base_silver = f"abfss://squad1@{adls_storage_account}.dfs.core.windows.net/dupla1/silver"
base_squad1 = f"abfss://squad1@{adls_storage_account}.dfs.core.windows.net"

caminhos = {
    "bronze_itens_pedido": f"{base_bronze}/ecommerce_itens_pedido",
    "bronze_rastreamento": f"{base_bronze}/ecommerce_rastreamento_entregas",
    "silver_itens_pedido": f"{base_silver}/ecommerce_itens_pedido",
    "silver_rastreamento": f"{base_silver}/ecommerce_rastreamento_entregas",
    "silver_pedidos": f"{base_squad1}/silver/ecommerce_pedidos",      # dependência (outra dupla)
    "silver_produtos": f"{base_squad1}/silver/ecommerce_produtos",    # dependência (outra dupla)
}

print("Caminhos configurados:")
for k, v in caminhos.items():
    print(f"  {k}: {v}")

In [0]:
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

credential = ClientSecretCredential(adls_tenant_id, adls_client_id, adls_client_secret)
service_client = DataLakeServiceClient(
    account_url=f"https://{adls_storage_account}.dfs.core.windows.net",
    credential=credential,
)
file_system_client_squad1 = service_client.get_file_system_client(file_system="squad1")

print("Procurando por 'pedidos' dentro de silver/ no container squad1:\n")
encontrou = False
for p in file_system_client_squad1.get_paths(path="", recursive=True):
    if p.is_directory and "silver" in p.name and "pedido" in p.name:
        print(f"  [DIR ] {p.name}")
        encontrou = True

if not encontrou:
    print("Nenhuma pasta Silver de pedidos encontrada ainda.")

In [0]:
caminhos["metadata_silver_control_log"] = f"abfss://squad1@{adls_storage_account}.dfs.core.windows.net/dupla1/metadata/silver_control_log"
print(f"Caminho log de controle Silver: {caminhos['metadata_silver_control_log']}")

In [0]:
def tabela_delta_existe(caminho):
    try:
        spark.read.format("delta").options(**adls_options).load(caminho).limit(1).count()
        return True
    except Exception:
        return False

print("Função de verificação de tabela Delta pronta.")

In [0]:
def carregar_referencias():
    df_silver_pedidos = (spark.read.format("delta").options(**adls_options)
        .load(caminhos["silver_pedidos"])
        .select("id_pedido").distinct())

    df_silver_produtos = (spark.read.format("delta").options(**adls_options)
        .load(caminhos["silver_produtos"])
        .select("sku", "preco_lista").distinct())

    return df_silver_pedidos, df_silver_produtos

print("Função de carga de referências pronta.")

In [0]:
def detectar_novo_bronze(nome_tabela, caminho_bronze):
    df_bronze = spark.read.format("delta").options(**adls_options).load(caminho_bronze)

    if tabela_delta_existe(caminhos["metadata_silver_control_log"]):
        df_control = spark.read.format("delta").options(**adls_options).load(caminhos["metadata_silver_control_log"])
        ja_processados = {
            r["arquivo_processado"] for r in
            df_control.filter(F.col("tabela_origem") == nome_tabela).select("arquivo_processado").collect()
        }
    else:
        ja_processados = set()

    df_novos = df_bronze.filter(~F.col("bronze_source_file").isin(ja_processados)) if ja_processados else df_bronze

    arquivos_no_lote = [r["bronze_source_file"] for r in df_novos.select("bronze_source_file").distinct().collect()]

    return df_novos, arquivos_no_lote

print("Função de detecção incremental (Bronze -> Silver) pronta.")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def aplicar_regras_itens_pedido(df_novos, df_silver_pedidos, df_silver_produtos, pedidos_disponivel):
    # Regra técnica 1: id_item_pedido não nulo nem duplicado (no lote + contra a Silver já gravada, se existir)
    janela_dedup_item = Window.partitionBy("id_item_pedido")
    nao_duplicado_no_lote = F.count("*").over(janela_dedup_item) == 1
    regra1 = F.col("id_item_pedido").isNotNull() & nao_duplicado_no_lote

    # Regra técnica 2: id_pedido deve existir em ecommerce_pedidos (FK)
    if pedidos_disponivel:
        ids_pedidos_validos = set(r["id_pedido"] for r in df_silver_pedidos.select("id_pedido").collect())
        regra2 = F.col("id_pedido").isin(ids_pedidos_validos) if ids_pedidos_validos else F.lit(False)
    else:
        regra2 = F.lit(None).cast("boolean")  # pendente: Silver de pedidos ainda não disponível

    # Regra técnica 3: sku deve existir em ecommerce_produtos (FK)
    skus_validos = set(r["sku"] for r in df_silver_produtos.select("sku").collect())
    regra3 = F.col("sku").isin(skus_validos) if skus_validos else F.lit(False)

    # Regra técnica 4: quantidade >= 1 e inteira
    regra4 = (F.col("quantidade") >= 1) & (F.col("quantidade") == F.floor(F.col("quantidade")))

    # Regra técnica 5: preco_unitario > 0
    regra5 = F.col("preco_unitario") > 0

    # Regra de negócio 6: desconto_aplicado não pode ser maior que preco_unitario
    regra6 = F.col("desconto_aplicado") <= F.col("preco_unitario")

    # Regra de negócio 8: desconto_aplicado não pode ser negativo
    regra8 = F.col("desconto_aplicado") >= 0

    # Regra de negócio 9: percentual de desconto não deve exceder 50% do preco_unitario
    # (limite oficial da regra é 50%; o detalhamento só explica que o uso legítimo real gira em torno de 20%)
    regra9 = F.col("desconto_aplicado") <= (F.col("preco_unitario") * 0.5)

    df_regras = (df_novos
        .withColumn("tecnica_1_pk_valida", regra1)
        .withColumn("tecnica_2_fk_pedido_valido", regra2)
        .withColumn("tecnica_3_fk_sku_valido", regra3)
        .withColumn("tecnica_4_quantidade_valida", regra4)
        .withColumn("tecnica_5_preco_valido", regra5)
        .withColumn("negocio_6_desconto_nao_maior_preco", regra6)
        .withColumn("negocio_8_desconto_nao_negativo", regra8)
        .withColumn("negocio_9_desconto_dentro_limite", regra9))

    # Regra de negócio 7 (soma dos itens bate com valor_total do pedido) depende da Silver de pedidos — pendente por enquanto
    if pedidos_disponivel:
        soma_por_pedido = (df_novos
            .withColumn("valor_item", (F.col("preco_unitario") - F.col("desconto_aplicado")) * F.col("quantidade"))
            .groupBy("id_pedido").agg(F.sum("valor_item").alias("soma_calculada"))
            .join(df_silver_pedidos.select("id_pedido", "valor_total"), on="id_pedido", how="left"))

        divergentes = soma_por_pedido.filter(
            F.col("valor_total").isNull() |
            (F.abs(F.col("soma_calculada") - F.col("valor_total")) > 0.01)
        ).count()
        print(f"    [REGRA 7] Pedidos com soma de itens divergente do valor_total: {divergentes}")
    else:
        print("    [REGRA 7] Pendente — Silver de pedidos ainda não disponível.")

    # Regra de negócio 10 (todo pedido deve ter >= 1 item) não é verificável aqui —
    # precisa ser checada do lado de "pedidos" (procurando quem não tem filho em itens_pedido).
    print("    [REGRA 10] Fora de escopo desta função — precisa ser checada a partir de 'pedidos', não de 'itens_pedido'.")

    return df_regras

print("Função de aplicação de regras para itens_pedido (Squad 1) pronta.")

In [0]:
STATUS_VALIDOS = ["em separacao", "coletado", "em transito", "saiu para entrega", "entregue"]

def aplicar_regras_rastreamento(df_novos, df_silver_pedidos, df_silver_rastreamento_existente, pedidos_disponivel):
    # (regras 1 a 6 continuam exatamente como antes — sem mudança)
    janela_dedup = Window.partitionBy("id_rastreamento")
    regra1 = F.col("id_rastreamento").isNotNull() & (F.count("*").over(janela_dedup) == 1)
    regra2 = F.col("status_entrega").isin(STATUS_VALIDOS)

    if pedidos_disponivel:
        ids_pedidos_validos = set(r["id_pedido"] for r in df_silver_pedidos.select("id_pedido").collect())
        regra3 = F.col("id_pedido_ecommerce").isNotNull() & (
            F.col("id_pedido_ecommerce").isin(ids_pedidos_validos) if ids_pedidos_validos else F.lit(False)
        )
    else:
        regra3 = F.lit(None).cast("boolean")

    regra4 = F.col("dt_evento").isNotNull() & (F.col("dt_evento") <= F.current_timestamp())
    regra5 = F.col("codigo_rastreio").rlike(r"^TRK\d{6}$")

    if df_silver_rastreamento_existente is not None:
        historico = df_silver_rastreamento_existente.select("id_pedido_ecommerce", "dt_evento")
        pedidos_do_lote = [r["id_pedido_ecommerce"] for r in df_novos.select("id_pedido_ecommerce").distinct().collect()]
        historico_relevante = historico.filter(F.col("id_pedido_ecommerce").isin(pedidos_do_lote))
    else:
        historico_relevante = None

    df_novos_marcado = df_novos.withColumn("_e_novo", F.lit(True))
    combinado = (df_novos_marcado.unionByName(historico_relevante.withColumn("_e_novo", F.lit(False)), allowMissingColumns=True)
                 if historico_relevante is not None else df_novos_marcado)

    janela_ordem = Window.partitionBy("id_pedido_ecommerce").orderBy("dt_evento")
    combinado = combinado.withColumn("_dt_evento_anterior", F.lag("dt_evento").over(janela_ordem))
    combinado = combinado.withColumn(
        "negocio_6_ordem_cronologica",
        F.col("_dt_evento_anterior").isNull() | (F.col("dt_evento") >= F.col("_dt_evento_anterior"))
    )

    df_regras = (combinado.filter(F.col("_e_novo"))
        .drop("_e_novo", "_dt_evento_anterior")
        .withColumn("tecnica_1_pk_valida", regra1)
        .withColumn("tecnica_2_status_valido", regra2)
        .withColumn("tecnica_3_fk_pedido_valido", regra3)
        .withColumn("tecnica_4_data_valida", regra4)
        .withColumn("tecnica_5_codigo_rastreio_valido", regra5))

    # Regra 8 (SLA coletado -> entregue <= 30 dias) — sem mudança
    eventos_coletado = df_regras.filter(F.col("status_entrega") == "coletado").groupBy("id_pedido_ecommerce").agg(F.min("dt_evento").alias("dt_coletado"))
    eventos_entregue = df_regras.filter(F.col("status_entrega") == "entregue").select("id_pedido_ecommerce", F.col("dt_evento").alias("dt_entregue"))
    prazos = eventos_coletado.join(eventos_entregue, on="id_pedido_ecommerce", how="inner").withColumn("dias_entrega", F.datediff("dt_entregue", "dt_coletado"))
    fora_do_prazo = prazos.filter(F.col("dias_entrega") > 30).count()
    print(f"    [REGRA 8] Pedidos entregues fora do prazo de 30 dias no lote: {fora_do_prazo}")

    if pedidos_disponivel:
        # Todos os eventos "entregue" já conhecidos (lote atual + histórico relevante)
        eventos_entregue_completo = combinado.filter(F.col("status_entrega") == "entregue").select("id_pedido_ecommerce").distinct()
        ids_com_evento_entregue = set(r["id_pedido_ecommerce"] for r in eventos_entregue_completo.collect())

        # Regra 7: pedido com status_pedido = 'Entregue' precisa ter evento 'entregue' no rastreamento
        pedidos_entregues = df_silver_pedidos.filter(F.col("status_pedido") == "Entregue").select("id_pedido").distinct()
        sem_evento_entregue = pedidos_entregues.filter(~F.col("id_pedido").isin(ids_com_evento_entregue)).count()
        print(f"    [REGRA 7] Pedidos com status 'Entregue' sem evento 'entregue' no rastreamento: {sem_evento_entregue}")

        # Regra 9: pedido cancelado não deve ter NENHUM evento de rastreamento
        pedidos_do_lote_ids = set(r["id_pedido_ecommerce"] for r in df_novos.select("id_pedido_ecommerce").distinct().collect())
        pedidos_cancelados_no_lote = set(
            r["id_pedido"] for r in
            df_silver_pedidos.filter((F.col("status_pedido") == "Cancelado") & (F.col("id_pedido").isin(pedidos_do_lote_ids)))
            .select("id_pedido").collect()
        )
        violacoes_cancelado = len(pedidos_cancelados_no_lote)  # qualquer pedido cancelado que apareceu no lote já é violação
        print(f"    [REGRA 9] Eventos de rastreamento recebidos para pedidos cancelados neste lote: {violacoes_cancelado}")

        # Regra 10, agora refinada: exclui cancelados do cálculo de consistência de transportadora
        ids_cancelados_geral = set(r["id_pedido"] for r in df_silver_pedidos.filter(F.col("status_pedido") == "Cancelado").select("id_pedido").collect())
        transportadoras_por_pedido = (df_regras.filter(~F.col("id_pedido_ecommerce").isin(ids_cancelados_geral))
            .groupBy("id_pedido_ecommerce").agg(F.countDistinct("id_transportadora").alias("qtd_transportadoras")))
        inconsistentes = transportadoras_por_pedido.filter(F.col("qtd_transportadoras") > 1).count()
        print(f"    [REGRA 10] Pedidos não cancelados com mais de 1 transportadora no lote: {inconsistentes}")
    else:
        print("    [REGRA 7] Pendente — Silver de pedidos ainda não disponível.")
        print("    [REGRA 9] Pendente — Silver de pedidos ainda não disponível.")
        transportadoras_por_pedido = df_regras.groupBy("id_pedido_ecommerce").agg(F.countDistinct("id_transportadora").alias("qtd_transportadoras"))
        inconsistentes = transportadoras_por_pedido.filter(F.col("qtd_transportadoras") > 1).count()
        print(f"    [REGRA 10] Pedidos com mais de 1 transportadora no lote (checagem parcial, sem excluir cancelados): {inconsistentes}")

    return df_regras

print("Função de aplicação de regras para rastreamento_entregas (Squad 1, completa) pronta.")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType
from datetime import datetime

schema_log = StructType([
    StructField("tabela_origem", StringType(), False),
    StructField("arquivo_processado", StringType(), False),
    StructField("total_linhas", LongType(), False),
    StructField("status", StringType(), False),
    StructField("timestamp_processamento", StringType(), False),
])

def processar_silver(df_regras, nome_tabela, caminho_silver):
    if df_regras is None:
        print(f"    Nenhum arquivo novo para '{nome_tabela}'.")
        return

    total = df_regras.count()
    resumo_por_arquivo = df_regras.groupBy("bronze_source_file").agg(F.count("*").alias("linhas")).collect()
    print(f"    >>> {len(resumo_por_arquivo)} arquivo(s) de origem promovido(s) para Silver de '{nome_tabela}' ({total} linhas)")

    df_final = df_regras.withColumn("silver_processed_at", F.current_timestamp())

    (df_final.write
        .format("delta")
        .options(**adls_options)
        .mode("append")
        .save(caminho_silver))

    print(f"    [OK] Silver de '{nome_tabela}' gravada.")

    try:
        agora = str(datetime.now())
        log_data = [(nome_tabela, r["bronze_source_file"], int(r["linhas"]), "SUCCESS", agora) for r in resumo_por_arquivo]
        df_log = spark.createDataFrame(log_data, schema=schema_log)

        (df_log.write
            .format("delta")
            .options(**adls_options)
            .mode("append")
            .save(caminhos["metadata_silver_control_log"]))

        print(f"    [OK] Log de controle Silver atualizado para '{nome_tabela}'.")
    except Exception as e_log:
        print(f"    [ERRO NO LOG] {type(e_log).__name__}: {e_log}")

print("Função de gravação na Silver pronta.")

In [0]:
import time
from datetime import datetime, timedelta

intervalo_segundos = 15
duracao_maxima_segundos = 90
inicio = datetime.now()
fim_previsto = inicio + timedelta(seconds=duracao_maxima_segundos)
falhas_consecutivas = 0
MAX_FALHAS_CONSECUTIVAS = 5

print(f"Iniciando processamento Silver às {inicio.strftime('%H:%M:%S')}.")
print(f"Vai rodar até {fim_previsto.strftime('%H:%M:%S')} (ou até interromper manualmente).\n")

while datetime.now() < fim_previsto:
    try:
        print(f"--- Verificação às {datetime.now().strftime('%H:%M:%S')} ---")

        # Referências compartilhadas (recarregadas a cada verificação, pra pegar dado novo se existir)
        pedidos_disponivel = tabela_delta_existe(caminhos["silver_pedidos"])
        df_silver_pedidos = (spark.read.format("delta").options(**adls_options).load(caminhos["silver_pedidos"])
                              if pedidos_disponivel else None)
        df_silver_produtos = (spark.read.format("delta").options(**adls_options)
                               .load(caminhos["silver_produtos"]).select("sku").distinct())

        if not pedidos_disponivel:
            print("    [AVISO] Silver de pedidos ainda indisponível — regras dependentes ficarão pendentes.")

        # --- ecommerce_itens_pedido ---
        df_novos_itens, arquivos_itens = detectar_novo_bronze("ecommerce_itens_pedido", caminhos["bronze_itens_pedido"])
        if arquivos_itens:
            df_regras_itens = aplicar_regras_itens_pedido(df_novos_itens, df_silver_pedidos, df_silver_produtos, pedidos_disponivel)
            processar_silver(df_regras_itens, "ecommerce_itens_pedido", caminhos["silver_itens_pedido"])
        else:
            print("    Nenhum arquivo novo para 'ecommerce_itens_pedido'.")

        # --- ecommerce_rastreamento_entregas ---
        rastreamento_existe = tabela_delta_existe(caminhos["silver_rastreamento"])
        df_silver_rastreamento_existente = (spark.read.format("delta").options(**adls_options).load(caminhos["silver_rastreamento"])
                                             if rastreamento_existe else None)

        df_novos_rastreamento, arquivos_rastreamento = detectar_novo_bronze("ecommerce_rastreamento_entregas", caminhos["bronze_rastreamento"])
        if arquivos_rastreamento:
            df_regras_rastreamento = aplicar_regras_rastreamento(df_novos_rastreamento, df_silver_pedidos, df_silver_rastreamento_existente, pedidos_disponivel)
            processar_silver(df_regras_rastreamento, "ecommerce_rastreamento_entregas", caminhos["silver_rastreamento"])
        else:
            print("    Nenhum arquivo novo para 'ecommerce_rastreamento_entregas'.")

        falhas_consecutivas = 0

    except KeyboardInterrupt:
        print("\nProcessamento interrompido manualmente.")
        break
    except Exception as e:
        falhas_consecutivas += 1
        print(f"    [ERRO] Falha na verificação ({falhas_consecutivas}/{MAX_FALHAS_CONSECUTIVAS}): {e}")
        if falhas_consecutivas >= MAX_FALHAS_CONSECUTIVAS:
            print(f"    {MAX_FALHAS_CONSECUTIVAS} falhas seguidas — interrompendo o loop.")
            break

    time.sleep(intervalo_segundos)

print(f"\nProcessamento finalizado às {datetime.now().strftime('%H:%M:%S')}.")